# 1.2 - CNN Inference

## 1: Dependencies and Imports

In [1]:
# Standard library
import os

# Third-party
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

## 2: Load Data

In [2]:
# Device configuration (use GPU if available, otherwise fallback to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# Model Architecture
class SimpleCNN2Digits(nn.Module):
    """
    CNN architecture matching the saved .pth weights.
    Designed for 28x56 input images.
    """
    def __init__(self):
        super(SimpleCNN2Digits, self).__init__()
        # Convolutional layers with 16 and 32 filters
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Input features: 32 channels * 7 height * 14 width = 3136
        self.fc1 = nn.Linear(32 * 7 * 14, 128)
        self.fc2 = nn.Linear(128, 100) # 100 classes for digits 00-99

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) 
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
model = SimpleCNN2Digits().to(device)

In [4]:
# Model loading
model_path = "../models/simple_cnn_2digits.pth"

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("Model loaded and ready for inference.")
else:
    print(f"Error: Model file not found at {model_path}")


Model loaded and ready for inference.


In [5]:
# Directory containing the input datasets
data_dir = "../data/raw_input"

try:
    # A. Tabular dataset (CSV)
    tabular_df = pd.read_csv(os.path.join(data_dir, "m_tabular.csv"))
    
    # B. Customer images (PyTorch .pt file)
    # Loading dictionary: {'order_id': [...], 'images': tensor_uint8}
    customer_data = torch.load(os.path.join(data_dir, "m_cust_images.pt"))
    
    # C. Seller images (PyTorch .pt file)
    seller_data = torch.load(os.path.join(data_dir, "m_sell_images.pt"))
    
    print("\nDatasets loaded successfully:")
    print(f"- Tabular: {tabular_df.shape}")
    print(f"- Customer Images Tensor: {customer_data['images'].shape}")
    print(f"- Seller Images Tensor: {seller_data['images'].shape}")

except FileNotFoundError as e:
    print(f"Error loading data: {e}")


Datasets loaded successfully:
- Tabular: (119143, 40)
- Customer Images Tensor: torch.Size([119143, 1, 28, 56])
- Seller Images Tensor: torch.Size([119143, 1, 28, 56])


## 3: Inference

In [6]:
def infer_states(data_dict, model, batch_size=128):
    """
    Performs batch inference on stored image tensors and returns a DataFrame of predictions.
    """
    model.eval()
    all_preds = []
    
    # Extract images and IDs from the data dictionary
    images_uint8 = data_dict.get('images', data_dict.get('imagenes'))
    order_ids = data_dict['order_id']
    
    total_records = len(order_ids)
    print(f"Starting inference on {total_records} records...")
    
    with torch.no_grad():
        for i in range(0, total_records, batch_size):
            # Slice the batch and normalize to [0, 1]
            batch_uint8 = images_uint8[i : i + batch_size]
            batch_float = batch_uint8.to(torch.float32) / 255.0
            
            # Ensure 4D tensor shape: [Batch, Channels, Height, Width]
            if batch_float.ndim == 3:
                batch_float = batch_float.unsqueeze(1)
            elif batch_float.ndim == 5:
                batch_float = batch_float.squeeze(1)
            
            batch_float = batch_float.to(device)
            
            # Forward pass and class selection
            outputs = model(batch_float)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            
            # Business Logic: Assign -1 to completely black images (empty entries)
            pixel_sums = batch_uint8.reshape(batch_uint8.size(0), -1).sum(dim=1)
            for j, pixel_sum in enumerate(pixel_sums):
                if pixel_sum == 0:
                    preds[j] = -1
            
            all_preds.extend(preds)

    print(f"Finished: {len(all_preds)} records processed successfully.\n")

    return pd.DataFrame({
        'order_id': order_ids,
        'state_num_pred': all_preds
    })

# Execute inference for both customer and seller datasets
customer_preds_df = infer_states(customer_data, model)
seller_preds_df = infer_states(seller_data, model)

Starting inference on 119143 records...
Finished: 119143 records processed successfully.

Starting inference on 119143 records...
Finished: 119143 records processed successfully.



## 4: Data Reintegration and Cleaning

In [7]:
# Column Injection

# 1. Deduplicate predictions to ensure a 1:1 mapping with order_id
# This prevents errors if an order contained multiple products
customer_preds_dict = customer_preds_df.drop_duplicates('order_id').set_index('order_id')['state_num_pred']
seller_preds_dict = seller_preds_df.drop_duplicates('order_id').set_index('order_id')['state_num_pred']

# 2. Map predictions to the main dataframe using order_id as the key
tabular_df['customer_state_num_pred'] = tabular_df['order_id'].map(customer_preds_dict)
tabular_df['seller_state_num_pred'] = tabular_df['order_id'].map(seller_preds_dict)

# 3. Handle missing values and enforce integer types
tabular_df['customer_state_num_pred'] = tabular_df['customer_state_num_pred'].fillna(-1).astype(int)
tabular_df['seller_state_num_pred'] = tabular_df['seller_state_num_pred'].fillna(-1).astype(int)

# Create final merged dataframe
final_df = tabular_df.copy()

print("Columns successfully injected.")
print(f"Final dimensions: {final_df.shape}") 

# Quick verification of injected columns
display(final_df[['order_id', 'customer_state_num', 'customer_state_num_pred', 'seller_state_num_pred']].head())

Columns successfully injected.
Final dimensions: (119143, 42)


,order_id,customer_state_num,customer_state_num_pred,seller_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,25,25,25
1,e481f51cbdc54678b7cc49136f2d6af7,25,25,25
2,e481f51cbdc54678b7cc49136f2d6af7,25,25,25
3,53cdb2fc8bc7dce0b6741e2150273451,4,4,26
4,47770eb9100c2d0c44946d9cf07ec65d,8,8,25


In [8]:
# Intergity check with missing value counts

# Count instances where geolocation could not be detected (marked as -1)
missing_customer_preds = (final_df['customer_state_num_pred'] == -1).sum()
missing_seller_preds = (final_df['seller_state_num_pred'] == -1).sum()

print("Integrity Summary:")
print(f"- Total orders processed: {len(final_df)}")
print(f"- Orders without detected geolocation (Customer): {missing_customer_preds}")
print(f"- Orders without detected geolocation (Seller): {missing_seller_preds}")

Integrity Summary:
- Total orders processed: 119143
- Orders without detected geolocation (Customer): 0
- Orders without detected geolocation (Seller): 833


## 5: Data Saving

In [9]:
# Define the final export path
final_path = '../data/Integrated.pkl'
os.makedirs('../data', exist_ok=True)

# Save the integrated dataset as a pickle file
# This preserves data types, including tensors and mapped categories
final_df.to_pickle(final_path)

print(f"Process completed. Master dataset saved at: {final_path}")
print("The file contains the original data combined with the CNN state predictions.")

Process completed. Master dataset saved at: ../data/Integrated.pkl
The file contains the original data combined with the CNN state predictions.
